# DeBERTA + LoRA fine tuning

## 1. Setup & Imports

In [ ]:
!pip install -q peft
!pip uninstall torchao -y

import os, re, random, warnings
os.environ["WANDB_MODE"] = "disabled"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
from collections import Counter
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'


## 2. Data Loading & EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option'); plt.ylabel('Count')
plt.show()

def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)
test_df['option_set'] = test_df.apply(option_set_key, axis=1)

dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Duplicate option-sets in train: {dup_count}")

train_option_sets = set(train_df['option_set'].unique())
test_matched = test_df['option_set'].isin(train_option_sets)
print(f"Test questions matched to train: {test_matched.sum()} / {len(test_df)} ({test_matched.mean()*100:.1f}%)")

## 3. Data Preprocessing & Strict Splitting

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d: uf.union(i, d[k])
        else: d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero group overlap)")

## 4. Model Inference Strategy

The top-3 most probable options from the model are used

## 5. Metric: MAP@3

In [ ]:
def map3_from_probs(probs, true_idx):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    true_lab = [LABELS[i] for i in true_idx]
    pred_lab = [" ".join(LABELS[j] for j in row) for row in top3]
    score = 0.0
    for t, p in zip(true_lab, pred_lab):
        for i, c in enumerate(p.split()[:3]):
            if c == t:
                score += 1.0/(i+1); break
    return score / len(true_idx)

## 6. DeBERTa-v3 + LoRA Fine-Tuning (3-Fold)

GroupKFold ensures zero data leakage between folds

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_len = []
for _, row in train_split.sample(min(300, len(train_split)), random_state=SEED).iterrows():
    p = clean_prompt(row['prompt'])
    for l in LABELS:
        sample_len.append(len(tokenizer(p, row[l])['input_ids']))
MAX_LEN = min(384, int(np.percentile(sample_len, 95)) // 32 * 32 + 32)
print(f"DeBERTa max_length: {MAX_LEN}")

class MCQDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = clean_prompt(row['prompt'])
        ids, mask = [], []
        for l in LABELS:
            enc = tokenizer(p, str(row[l]), max_length=MAX_LEN, padding='max_length',
                            truncation=True, return_tensors='pt')
            ids.append(enc['input_ids'].squeeze(0))
            mask.append(enc['attention_mask'].squeeze(0))
        item = {'input_ids': torch.stack(ids), 'attention_mask': torch.stack(mask)}
        if 'label' in row:
            item['labels'] = torch.tensor(row['label'])
        return item

In [ ]:
!pip install -q torchao>=0.16.0 --quiet

EPOCHS = 5
LR = 2e-5
BATCH_SIZE = 4
GRAD_ACCUM = 2
N_FOLDS = 3
GROUP_NAME = "deberta_lora"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

gkf = GroupKFold(n_splits=N_FOLDS)
val_probs_list, test_probs_list = [], []
oof_probs = np.zeros((len(train_split), 5))

for fold, (tr_i, va_i) in enumerate(gkf.split(train_split, y_tr, groups)):
    wandb.init(
        project="smart-mcq-solver",
        entity="23f2004192-dl-genai-project",
        group=GROUP_NAME,
        job_type="cv_fold",
        name=f"fold_{fold}",
        config={"epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN,
                "batch_size": BATCH_SIZE, "lora_r": LORA_R,
                "lora_alpha": LORA_ALPHA, "model": MODEL_NAME}
    )

    tr_df = train_split.iloc[tr_i]
    va_df = train_split.iloc[va_i]
    tr_loader = DataLoader(MCQDataset(tr_df), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(MCQDataset(va_df), batch_size=BATCH_SIZE, shuffle=False)

    base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["query_proj", "value_proj", "key_proj"],
        bias="none"
    )
    model = get_peft_model(base_model, lora_config).to(device)
    model.print_trainable_parameters()

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = (len(tr_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    best_map3, best_state = 0, None
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(tr_loader, desc=f"F{fold} E{epoch}")):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            epoch_loss += out.loss.item()
            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        all_probs, all_true = [], []
        with torch.no_grad():
            for batch in va_loader:
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(input_ids=ids, attention_mask=mask).logits
                all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
                all_true.extend(batch['labels'].cpu().numpy())
        probs_va = np.vstack(all_probs)
        val_map3 = map3_from_probs(probs_va, np.array(all_true))
        val_preds = probs_va.argmax(axis=1)
        val_acc = accuracy_score(np.array(all_true), val_preds)
        val_f1 = f1_score(np.array(all_true), val_preds, average='macro')

        wandb.log({"epoch": epoch, "train_loss": epoch_loss / len(tr_loader),
                   "val_map3": val_map3, "val_accuracy": val_acc, "val_f1": val_f1})
        print(f"  F{fold} E{epoch}: loss={epoch_loss/len(tr_loader):.4f} acc={val_acc:.4f} MAP3={val_map3:.4f}")

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    wandb.finish()

    model.eval()
    with torch.no_grad():
        probs = []
        for batch in DataLoader(MCQDataset(va_df), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        oof_probs[va_i] = np.vstack(probs)

        probs = []
        for batch in DataLoader(MCQDataset(val_split), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        val_probs_list.append(np.vstack(probs))

        probs = []
        for batch in DataLoader(MCQDataset(test_df), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        test_probs_list.append(np.vstack(probs))

    del model, base_model
    torch.cuda.empty_cache()

val_probs_deberta = np.mean(val_probs_list, axis=0)
test_probs_deberta = np.mean(test_probs_list, axis=0)
val_true = val_split['label'].values

deberta_val_map3 = map3_from_probs(val_probs_deberta, val_true)
print(f"\nDeBERTa+LoRA Val MAP3: {deberta_val_map3:.4f}")


## 7. Inference: Top-3 from DeBERTa+LoRA

Generate top-3 predictions directly from the model’s softmax probabilities.

In [ ]:
def make_submission(probs):
    """Generate top-3 predictions from model probabilities."""
    preds = []
    for i in range(len(probs)):
        top3 = np.argsort(-probs[i])[:3]
        preds.append(' '.join(LABELS[j] for j in top3))
    return preds

test_preds = make_submission(test_probs_deberta)
print(f"Generated {len(test_preds)} predictions")


## 8. Evaluation on Validation Set

In [ ]:
val_preds_idx = np.argmax(val_probs_deberta, axis=1)
acc = accuracy_score(val_true, val_preds_idx)
f1 = f1_score(val_true, val_preds_idx, average='macro')
val_map3 = map3_from_probs(val_probs_deberta, val_true)

print(f"DeBERTa+LoRA Val Accuracy: {acc:.4f}")
print(f"DeBERTa+LoRA Val Macro F1: {f1:.4f}")
print(f"DeBERTa+LoRA Val MAP@3:   {val_map3:.4f}")

wandb.init(
    project="smart-mcq-solver",
    entity="23f2004192-dl-genai-project",
    group=GROUP_NAME,
    job_type="summary",
    name="final_summary",
    tags=["deberta-lora", "final"],
    config={"epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN,
            "lora_r": LORA_R, "lora_alpha": LORA_ALPHA}
)
wandb.log({"final_val_map3": val_map3, "final_val_accuracy": acc,
           "final_val_f1": f1})
wandb.finish()


## 9. Final Submission

In [ ]:
sub = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': test_preds
}).sort_values('ID').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)
print("Submission saved")
print(sub.head(10))